# Week 03 — Hidden Pairs

This baseline predicts whether a hidden pair belongs to the positive class (`target = 1`) using 512 anonymous numerical features.

We fit one small decision tree on 2,520 training rows, evaluate it on the untouched 560-row validation set, and use the 1,116-row test set only to generate probabilities. This is an educational baseline, not a leaderboard-optimized solution.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

To use Colab, upload this notebook together with `train.csv`, `validation.csv`, `test.csv`, and `sample_submission.csv`, then run all cells. No internet or GPU is required after the files are uploaded.

## 1. Imports and reproducibility

Instructor comment preserved: explain whether PyTorch is used and which libraries are needed. Here, pandas handles CSV tables, NumPy supports numerical checks, and scikit-learn supplies the model and ROC-AUC. PyTorch is not used because the supplied files already contain the 512 numerical features.

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier

SEED = 20260916
random.seed(SEED)
np.random.seed(SEED)
notebook_started = time.perf_counter()

## 2. Load the supplied files

The partitions are already prepared. Do not create another random split, combine partitions, or fit anything on test data.

In [ ]:
# Instructor comment preserved: DATA_DIR is the directory containing the four supplied CSV files.
DATA_DIR = Path("data/datasets/hidden-pairs")

train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape)
print("validation:", validation.shape)
print("test:", test.shape)
print("sample submission:", sample_submission.shape)

## 3. Validate the input contract

Check the expected row counts, required columns, missing values, binary targets, and identical feature order before modeling.

In [ ]:
assert train.shape == (2520, 514)
assert validation.shape == (560, 514)
assert test.shape == (1116, 513)
assert sample_submission.shape == (1116, 2)

assert {"row_id", "target"}.issubset(train.columns)
assert {"row_id", "target"}.issubset(validation.columns)
assert "row_id" in test.columns and "target" not in test.columns
assert list(sample_submission.columns) == ["row_id", "target"]

assert not train.isna().any().any()
assert not validation.isna().any().any()
assert not test.isna().any().any()
assert set(train["target"].unique()) == {0, 1}
assert set(validation["target"].unique()) == {0, 1}

## 4. Select features

`row_id` aligns rows and `target` is the answer. Neither is a model feature. The same 512 ordered feature columns must occur in every partition.

### Task 1: select feature columns

Create `feature_columns` from `train.columns`, excluding `row_id` and `target`. The available variables are the columns of `train`. The result represents the ordered model inputs and must be a list containing exactly 512 names.

In [ ]:
feature_columns = [column for column in train.columns if column not in {"row_id", "target"}]


In [ ]:
assert len(feature_columns) == 512
assert not {"row_id", "target"}.intersection(feature_columns)
assert feature_columns == [column for column in validation.columns if column not in {"row_id", "target"}]
assert feature_columns == [column for column in test.columns if column != "row_id"]

## 5. Construct model inputs

Keep validation untouched by training. The tree receives the supplied numerical columns directly, so there is no preprocessing step to fit.

### Task 2: construct matrices and targets

Using `train`, `validation`, `test`, and `feature_columns`, create `X_train`, `y_train`, `X_validation`, `y_validation`, and `X_test`. These represent the three feature matrices and two available target vectors. Every feature matrix must have 512 columns and each target must match its labeled partition.

In [ ]:
X_train = train[feature_columns]
y_train = train["target"]
X_validation = validation[feature_columns]
y_validation = validation["target"]
X_test = test[feature_columns]


In [ ]:
assert X_train.shape == (2520, 512)
assert X_validation.shape == (560, 512)
assert X_test.shape == (1116, 512)
assert len(y_train) == len(X_train)
assert len(y_validation) == len(X_validation)

## 6. Define and train the baseline

A depth-3 tree is transparent and fast. `min_samples_leaf=20` prevents rules based on very small groups. It is fitted on training rows only.

### Task 3: create the model

Create `model` as a `DecisionTreeClassifier` with `max_depth=3`, `min_samples_leaf=20`, and `random_state=SEED`. The result must be an unfitted classifier.

In [ ]:
model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=SEED)


### Task 4: fit the model

Fit `model` using only `X_train` and `y_train`. The result should represent a classifier learned only from training rows and must expose `predict_proba`.

In [ ]:
model.fit(X_train, y_train)


## 7. Evaluate on validation

ROC-AUC measures how well continuous scores rank positive rows above negative rows across all thresholds. Thresholded class labels discard ranking information and should not be used for ROC-AUC.

### Task 5: validation probabilities and ROC-AUC

Use `model`, `X_validation`, and `y_validation` to create `validation_probability` for class 1 and scalar `validation_auc`. There must be one probability per validation row, and the AUC must be finite and approximately 0.7473.

In [ ]:
validation_probability = model.predict_proba(X_validation)[:, 1]
validation_auc = roc_auc_score(y_validation, validation_probability)


In [ ]:
assert validation_probability.shape == (len(validation),)
assert np.isfinite(validation_auc)
assert np.isclose(validation_auc, 0.7473086734693878)
assert 0.0 <= validation_auc <= 1.0

## 8. Predict the test set

The test set remains unlabeled and is used only after fitting. Repeating prediction verifies deterministic behavior. The instructor reference public score is approximately 0.72157; public labels are not available to or accessed by this notebook.

### Task 6: test probabilities

Use `model` and `X_test` to create `test_probability` for class 1. Repeat the prediction as `test_probability_repeat`; both arrays must be identical and contain one probability in `[0, 1]` per test row.

In [ ]:
test_probability = model.predict_proba(X_test)[:, 1]
test_probability_repeat = model.predict_proba(X_test)[:, 1]


## 9. Build and verify the submission

Starting from the sample submission preserves the required row order and schema. Replace only its `target` column with positive-class probabilities.

### Task 7: fill and save the submission

Copy `sample_submission` into `submission`, replace its target values with `test_probability`, verify the two-column schema, row count, exact test ID order, unique IDs, missing values, and probability range, then save `submission.csv` with `index=False`.

In [ ]:
submission = sample_submission.copy()
submission["target"] = test_probability

assert list(submission.columns) == ["row_id", "target"]
assert len(submission) == len(test)
assert submission["row_id"].tolist() == test["row_id"].tolist()
assert submission["row_id"].is_unique
assert not submission.isna().any().any()
assert submission["target"].between(0, 1).all()

submission.to_csv("submission.csv", index=False)


In [ ]:
runtime_seconds = time.perf_counter() - notebook_started
print(f"Validation ROC-AUC: {validation_auc:.6f}")
print(f"Runtime: {runtime_seconds:.2f} seconds")
print(submission.head())

## Task 8 — compact model comparison

Run the supplied train/validation split with the baseline tree, scaled logistic regression, random forest, and histogram gradient boosting. Validation is for comparison only; none of the models is fitted on it.


### Task 8 solution — model comparison

In [ ]:
from time import perf_counter
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidates = {
    "Decision tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=SEED),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    "Random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=15, random_state=SEED),
}

comparison_rows = []
for name, estimator in candidates.items():
    started = perf_counter()
    estimator.fit(X_train, y_train)
    elapsed = perf_counter() - started
    probability = estimator.predict_proba(X_validation)[:, 1]
    comparison_rows.append({
        "model": name,
        "validation ROC-AUC": roc_auc_score(y_validation, probability),
        "fit seconds": elapsed,
    })

comparison = pd.DataFrame(comparison_rows).sort_values("validation ROC-AUC", ascending=False)
display(comparison.round({"validation ROC-AUC": 4, "fit seconds": 3}))

selected_name = comparison.iloc[0]["model"]
print(f"Selected by validation ROC-AUC: {selected_name}")
